# Projet 5MA-OSI : Transfert d'organes en domino sous incertitudes

## Introduction

Malgré l'augmentation croissante du nombre de transplantations d'organes effectuées chaque année (environ 6000 en 2017 dont 3782 transplantations de reins), la demande reste en perpétuelle augmentation. Ainsi 6000 organes, dont 3782 reins, ont été transplantés en 2017, mais il y avait encore 24000 personnes en attente d'un organe la même année. Les organes transplantés peuvent provenir d'un donneur décédé ou, dans le cas des reins et du foie, d'un donneur vivant consentant, le plus souvent membre de la famille du patient. Hélas, même si un proche accepte de prendre ce risque pour sa santé, il ne sera pas forcément compatible avec le patient. Pour cette raison, les pratiques médicales et les législations évoluent dans de nombreux pays afin de permettre la mise en place d'un programme d'échange de dons d'organes.

L'exemple le plus simple d'échange de don d'organes est celui où deux patients $P_1$ et $P_2$ sont accompagnés de donneurs $D_1$ et $D_2$. Les patients sont supposés incompatibles avec les donneurs qui les accompagnent, mais on suppose que $D_1$ est compatible avec $P_2$ et $D_2$ avec $P_1$. Il est alors possible de transplanter un organe de $D_1$ vers $P_2$ et de $D_2$ vers $P_1$ avec le consentement de tous et en suivant la procédure légale.

Plus généralement, un cycle d'échange d'organes associe $k$ paires de patient-donneur $(P_{i_1},D_{i_1}), \dots, (P_{i_k},D_{i_k})$ de sorte que $D_{i_l}$ donne à $P_{i_{l+1}}$ pour $l=1,\dots,k-1$ et $D_{i_k}$ donne à $P_{i_1}$.
Par ailleurs, le point essentiel est que les transferts soient tous réalisés en même temps et dans le même hôpital pour éviter qu'une rétractation de dernière minute ne lèse un patient et son donneur, et que les patients et donneurs venus ensemble et leur famille puissent se soutenir émotionnellement durant l'hospitalisation. 
Pour cette raison, le nombre d'échanges prenant place au sein d'un même cycle est nécessairement limité. En pratique, l'organisation d'un cycle de trois paires est déjà une épreuve pour le personnel d'un hôpital, et le plus grand cycle ayant jamais eu lieu a a impliqué six patients et donneurs.

Dans ce projet, nous prendrons le point de vue de l'organisme national responsable de la gestion du programme d'échange d'organes. 
À chaque phase d'échange, l'objectif de cet organisme est de choisir un ensemble de cycles d'échanges entre paires compatibles afin de maximiser le nombre de patients recevant un organe. Dans certains cas, on peut aussi donner une priorité à certains patients en fonction de la gravité de leur état ou de la durée de leur attente. 
Pour cela, on pourra attribuer des poids différents à chaque patient et maximiser la somme des poids des patients recevant un organe. 

## Étude du problème déterministe


### Jeux de données

Tous vos tests seront basés sur les jeux de données de la [PrefLib](https://www.preflib.org/dataset/00036). Ces jeux de données ne correspondent pas à des programmes d'échanges d'organes réels pour des raisons de confidentialité, mais ils reproduisent la structure des données réelles plus fidèlement que les données aléatoires utilisées pour le projet de RO. Ils 
sont constituées d'informations individuelles et d'un graphe de compatibilité. Chaque fichier .wmd décrit un graphe de compatibilité orienté, $G=(V,A)$, où chaque sommet de $V$ représente une paire patient-donneur et où un arc entre deux paires $(P_k,D_k)$ et $(P_l,D_l)$ signifie que $D_k$ est compatible avec $P_l$. La compatibilité est obtenue à partir des données biologiques individuelles (e.g., les groupes sanguins) et d'un _test croisé_ lors duquel des biologistes mettent en présence des tissus d'un malade et d'un donneur supposé. Chaque fichier est composé comme suit :  
- les 11 premières ligne contiennent diverses informations sur le fichier, dont le nombre de sommets ($n$) et le nombre d'arcs ($m$),
- les $n$ lignes suivantes nomment les sommets (un sommet par paire),
- les $m$ lignes restantes contiennent les arcs du graphe et le poids de chaque arc. Le poids de chaque arc indique l'urgence de la situation du malade de la paire destination. 

Je fournis ci-dessous une fonction permettant de lire les fichiers .wmd et retournant un graphe de compatibilité pondéré par les poids de chaque patient. Pour vous aider à vous lancer, je fournis déjà quelques instances en accompagnement de ce sujet sur Moodle. Elles contiennent un nombre croissant de paires patient-donneur allant de 16 à 512.

In [2]:
# Include the necessary packages
using Random, Graphs, JuMP, HiGHS, DelimitedFiles, Distributions
include("KEP_readfile.jl")

read_dat_file

### Formulations compactes

Dans le cadre du TD/TP du cours de Recherche Opérationnelle, il vous a été demandé de coder trois formulations PLNE, à savoir :
- celle sans contrainte sur la longueur des cycles
- celle avec des cycles de taille 2
- celle avec la décomposition par hôpital pour des cycles de taille 2 à 6.

J'ai recopié la correction distribuée sur Moodle dans le fichier formulation_compactes.jl. Je l'inclus ci dessous. Afin de prendre le sujet en main, comparer les solutions des 3 modèles sur un ensemble de jeux de données bien choisis. Analyser les résultats d'un point de vue informatique, pratique et sociétal.

### Génération de colonnes

L'objectif de cette partie du projet est de coder une méthode de génération de colonnes pour le problème de dons d'organes en dominos (KEP pour Kidney Exchange Problem en anglais) sans incertitudes. Votre code s'appuiera sur les éléments vus en CM et leur application au KEP vue en TD. Pour compléter cette présentation, [une page de la documentation de JuMP](https://jump.dev/JuMP.jl/stable/tutorials/algorithms/cutting_stock_column_generation/) est dédiée à la génération de colonnes. Elle vous offre une autre entrée en matière sur la question.

1. Coder une fonction résolvant le modèle par cycles après énumération de touts les cycles de taille $K$ ou moins. Tester pour $K=2,3,4$.
2. Coder une méthode de génération de colonnes pour résoudre la relaxation linéaire de la formulation par cycles. Vous implémenterez deux versions de la fonction de _pricing_ (résolution du sous-problème) : une ou le sous-problème est formulé à l'aide d'un ensemble de PLNE, et une ou le sous-problème est résolu par un algorithme de plus long chemin.
3. Coder une fonction qui résout le problème de dons en dominos de façon approchée à l'aide de l'heuristique de génération de colonnes décrite en CM.


------

### Travail à réaliser

_(Cette partie n'est pas à rendre pour le 14 novembre. Elle ne sera commencée qu'après le premier CM de la partie stochastique, fin novembre.)_

Vous savez désormais récupérer des jeux de données pour le problème avec incertitudes à partir de la PrefLib et vous pouvez adapter ce travail pour en générer de nouveaux selon la méthode décrite au projet de 4A. L'objet de la suite de cette partie sera de coder les méthodes d'optimisation sous incertitudes suivantes.


1. Coder un algorithme permettant de simuler le processus réel en deux étapes, dans lequel l'étape de planification est réalisée à l'aide des méthodes développées dans le cadre déterministe.
2. Utiliser ces simulations pour étudier la sensibilité de la planification aux incertitudes en fonction de la taille maximum des cycles. Commentez.
3. En pratique, de nouvelles personnes entre et sortent du programme d'échange en continu. Donc des programmes sont régulièrement calculés et de nouvelles opérations programmés. Simuler un tel processus dynamique sur 6 mois en supposant que de nouvelles opérations peuvent être planifiées toutes les 2 semaines. Commentez.
4. Coder un modèle de maximisation de l'espérance du nombre de transferts à travers un calcul théorique des probabilités de réussite. Comparer l'utilisation de ce modèle avec les modèles déterministes dans le processus dynamique. Commentez.
5. Analyser d'un point de vue sociétal les résultats des différents algorithmes. Si besoin créer des jeux de données qui permettent d'illustrer certains comportements bénéfiques ou problématiques.
6. Sans la coder, dites ce que l'on peut attendre comme apport de la méthode avec recours demandée ci-dessous.

_Hors programme : la fin de l'énoncé ci-dessous ne fait plus partie du travail à réaliser pendant ce cours. Je la laisse, car j'y décris approche plus réaliste du problème de don en dominos._

1. Coder un modèle avec recours dans lequel un ensemble d'arcs à tester est calculé au premier niveau et les cycles à réaliser sont calculés au second niveau en fonction des résultats des tests croisés.
2. Améliorer le modèle précédent et la méthode utilisée pour le résoudre. On pourra essayer de nombreuses possibilités telles que : différentes formulations pour le problème de second niveau (cf la partie déterministe), la méthode L-Shaped, le renforcement du premier niveau par une contrainte de Jensen, le prétraitement du graphe de compatibilité pour éliminer des arcs/sommets "inutiles", la réduction a priori de l'ensemble d'arcs pour éliminer les arcs trop peu probables (penser aux enjeux éthiques), la restriction du problème à des petits cycles, etc.
3. Coder une approches averse aux risques : maximisation de la C-VaR ou optimisation robuste avec ensemble d'incertitudes de Bertsimas ("budgeted uncertainty")

Comme indiqué plus tôt, ces questions seront détaillées lors des séances de CM et de TD. Dans tous les cas, vous devrez définir un protocole de tests pour évaluer ces différentes méthodes et les comparer entre elles. Vous pourrez vous référer au travail effectué sur la partie déterministe pour construire et décrire clairement un protocole rigoureux qui permettra de répondre à un ensemble de questions importantes pour le problème réel. __Ce protocole, les résultats des tests qui en découleront et la discussion qui suivra sont les éléments permettront de montrer votre compréhension du cours et votre capacité à prendre du recul.__

## Question 1


In [3]:
include("solver.jl")
include("check_failure.jl")

Gtilde, edge_weight, gsm, gsd, pra = read_dat_file("data_KEP/KEP_191.dat"); # Loads a stochastic KEP problem  
test = KEP_test(Gtilde; SP_method="Bellmann") #  Create a determinist KEP problem
exchanges, nb_choosen = solve_KEP(test)
failure_rates = get_failure_rates(Gtilde, pra, "Binomial") # Compute exchanges failure rates 
successful_exchanges, nb_successful = check_exchange_failure(exchanges, failure_rates) # Check if choosen transfert are actually realisable

println("** 2 Steps stochastic resolution **")
println(" • Determinist part -> number of choosen exchanges : $nb_choosen")
println(" • Real compatibity check : \n\t -> number of successful exchanges : $nb_successful \n\t -> performed exchanges : $successful_exchanges")

** 2 Steps stochastic resolution **
 • Determinist part -> number of choosen exchanges : 352
 • Real compatibity check : 
	 -> number of successful exchanges : 30 
	 -> performed exchanges : [[87, 431], [418, 493], [6, 212], [273, 424], [118, 392], [144, 209], [327, 512], [159, 409], [154, 400], [142, 427], [265, 362], [81, 473], [14, 464], [257, 467], [225, 352]]


## Question 2

In [15]:
G, edge_weight, gsm, gsd, pra = read_dat_file("data_KEP/KEP_191.dat");

#Initialisation pour K=2
test_2 = KEP_test(G ; K=2, SP_method="Bellmann")
nb_exchanges_2 = Vector{Int}()
nb_successful_exchanges_binomial_2 = Vector{Int}()
nb_successful_exchanges_binomialunos_2 = Vector{Int}()
nb_successful_exchanges_binomialapd_2 = Vector{Int}()
nb_successful_exchanges_constant_2 = Vector{Int}()

#Initialisation pour K=3
test_3 = KEP_test(G ; K=3, SP_method="Bellmann")
nb_exchanges_3 = Vector{Int}()
nb_successful_exchanges_binomial_3 = Vector{Int}()
nb_successful_exchanges_binomialunos_3 = Vector{Int}()
nb_successful_exchanges_binomialapd_3 = Vector{Int}()
nb_successful_exchanges_constant_3 = Vector{Int}()

#Initialisation pour K=4
test_4 = KEP_test(G ; K=4, SP_method="Bellmann")
nb_exchanges_4 = Vector{Int}()
nb_successful_exchanges_binomial_4 = Vector{Int}()
nb_successful_exchanges_binomialunos_4 = Vector{Int}()
nb_successful_exchanges_binomialapd_4 = Vector{Int}()
nb_successful_exchanges_constant_4 = Vector{Int}()

for i in 1:500

    # New failure rates
    fr_binomial = get_failure_rates(G,pra,"Binomial")
    fr_binomialunos = get_failure_rates(G,pra,"BinomialUNOS")
    fr_binomialapd = get_failure_rates(G,pra,"BinomialAPD")
    fr_constant = get_failure_rates(G,pra,"Constant")

    #println(i)
    exchanges = solve_KEP(test_2)
    push!(nb_exchanges_2, exchanges[3])
    push!(nb_successful_exchanges_binomial_2, check_exchange_failure(exchanges[1], fr_binomial)[2])
    push!(nb_successful_exchanges_binomialunos_2, check_exchange_failure(exchanges[1], fr_binomialunos)[2])
    push!(nb_successful_exchanges_binomialapd_2, check_exchange_failure(exchanges[1], fr_binomialapd)[2])
    push!(nb_successful_exchanges_constant_2, check_exchange_failure(exchanges[1], fr_constant)[2])

    exchanges = solve_KEP(test_3)
    push!(nb_exchanges_3, exchanges[3])
    push!(nb_successful_exchanges_binomial_3, check_exchange_failure(exchanges[1], fr_binomial)[2])
    push!(nb_successful_exchanges_binomialunos_3, check_exchange_failure(exchanges[1], fr_binomialunos)[2])
    push!(nb_successful_exchanges_binomialapd_3, check_exchange_failure(exchanges[1], fr_binomialapd)[2])
    push!(nb_successful_exchanges_constant_3, check_exchange_failure(exchanges[1], fr_constant)[2])
    
    exchanges = solve_KEP(test_4)
    push!(nb_exchanges_4, exchanges[3])
    push!(nb_successful_exchanges_binomial_4, check_exchange_failure(exchanges[1], fr_binomial)[2])
    push!(nb_successful_exchanges_binomialunos_4, check_exchange_failure(exchanges[1], fr_binomialunos)[2])
    push!(nb_successful_exchanges_binomialapd_4, check_exchange_failure(exchanges[1], fr_binomialapd)[2])
    push!(nb_successful_exchanges_constant_4, check_exchange_failure(exchanges[1], fr_constant)[2])
end

InterruptException: InterruptException:

In [47]:
#K = 2
println("========================= K=2 =============================")
println("Nombre moyen de transferts réalisables : ", mean(nb_exchanges_2))
println("Nombre moyen de transferts réellement réalisés pour la distribution Binomial : ", mean(nb_successful_exchanges_binomial_2))
println("Variance des transferts réellement réalisés pour la distribution Binomial : ", var(nb_successful_exchanges_binomial_2))
println("Pourcentage de transferts réellement réalisés pour la distribution Binomial : ", 100*mean(nb_successful_exchanges_binomial_2)/mean(nb_exchanges_2))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution BinomialUNOS : ", mean(nb_successful_exchanges_binomialunos_2))
println("Variance des transferts réellement réalisés pour la distribution BinomialUNOS : ", var(nb_successful_exchanges_binomialunos_2))
println("Pourcentage de transferts réellement réalisés pour la distribution BinomialUNOS : ", 100*mean(nb_successful_exchanges_binomialunos_2)/mean(nb_exchanges_2))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution BinomialAPD : ", mean(nb_successful_exchanges_binomialapd_2))
println("Variance des transferts réellement réalisés pour la distribution BinomialAPD : ", var(nb_successful_exchanges_binomialapd_2))
println("Pourcentage de transferts réellement réalisés pour la distribution BinomialAPD : ", 100*mean(nb_successful_exchanges_binomialapd_2)/mean(nb_exchanges_2))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution Constant : ", mean(nb_successful_exchanges_constant_2))
println("Variance des transferts réellement réalisés pour la distribution Constant : ", var(nb_successful_exchanges_constant_2))
println("Pourcentage de transferts réellement réalisés pour la distribution Constant : ", 100*mean(nb_successful_exchanges_constant_2)/mean(nb_exchanges_2))


#K = 3
println("========================= K=3 =============================")
println("Nombre moyen de transferts réalisables : ", mean(nb_exchanges_3))
println("Nombre moyen de transferts réellement réalisés pour la distribution Binomial : ", mean(nb_successful_exchanges_binomial_3))
println("Variance des transferts réellement réalisés pour la distribution Binomial : ", var(nb_successful_exchanges_binomial_3))
println("Pourcentage de transferts réellement réalisés pour la distribution Binomial : ", 100*mean(nb_successful_exchanges_binomial_3)/mean(nb_exchanges_3))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution BinomialUNOS : ", mean(nb_successful_exchanges_binomialunos_3))
println("Variance des transferts réellement réalisés pour la distribution BinomialUNOS : ", var(nb_successful_exchanges_binomialunos_3))
println("Pourcentage de transferts réellement réalisés pour la distribution BinomialUNOS : ", 100*mean(nb_successful_exchanges_binomialunos_3)/mean(nb_exchanges_3))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution BinomialAPD : ", mean(nb_successful_exchanges_binomialapd_3))
println("Variance des transferts réellement réalisés pour la distribution BinomialAPD : ", var(nb_successful_exchanges_binomialapd_3))
println("Pourcentage de transferts réellement réalisés pour la distribution BinomialAPD : ", 100*mean(nb_successful_exchanges_binomialapd_3)/mean(nb_exchanges_3))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution Constant : ", mean(nb_successful_exchanges_constant_3))
println("Variance des transferts réellement réalisés pour la distribution Constant : ", var(nb_successful_exchanges_constant_3))
println("Pourcentage de transferts réellement réalisés pour la distribution Constant : ", 100*mean(nb_successful_exchanges_constant_3)/mean(nb_exchanges_3))


#K = 4
println("========================= K=4 =============================")
println("Nombre moyen de transferts réalisables : ", mean(nb_exchanges_4))
println("Nombre moyen de transferts réellement réalisés pour la distribution Binomial : ", mean(nb_successful_exchanges_binomial_4))
println("Variance des transferts réellement réalisés pour la distribution Binomial : ", var(nb_successful_exchanges_binomial_4))
println("Pourcentage de transferts réellement réalisés pour la distribution Binomial : ", 100*mean(nb_successful_exchanges_binomial_4)/mean(nb_exchanges_4))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution BinomialUNOS : ", mean(nb_successful_exchanges_binomialunos_4))
println("Variance des transferts réellement réalisés pour la distribution BinomialUNOS : ", var(nb_successful_exchanges_binomialunos_4))
println("Pourcentage de transferts réellement réalisés pour la distribution BinomialUNOS : ", 100*mean(nb_successful_exchanges_binomialunos_4)/mean(nb_exchanges_4))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution BinomialAPD : ", mean(nb_successful_exchanges_binomialapd_4))
println("Variance des transferts réellement réalisés pour la distribution BinomialAPD : ", var(nb_successful_exchanges_binomialapd_4))
println("Pourcentage de transferts réellement réalisés pour la distribution BinomialAPD : ", 100*mean(nb_successful_exchanges_binomialapd_4)/mean(nb_exchanges_4))
println("----------------------")
println("Nombre moyen de transferts réellement réalisés pour la distribution Constant : ", mean(nb_successful_exchanges_constant_4))
println("Variance des transferts réellement réalisés pour la distribution Constant : ", var(nb_successful_exchanges_constant_4))
println("Pourcentage de transferts réellement réalisés pour la distribution Constant : ", 100*mean(nb_successful_exchanges_constant_4)/mean(nb_exchanges_4))


========================= K=2 =============================
Nombre moyen de transferts réalisables : 342.0
Nombre moyen de transferts réellement réalisés pour la distribution Binomial : 30.748
Variance des transferts réellement réalisés pour la distribution Binomial : 57.90029659318637
Pourcentage de transferts réellement réalisés pour la distribution Binomial : 8.990643274853802
----------------------
Nombre moyen de transferts réellement réalisés pour la distribution BinomialUNOS : 180.096
Variance des transferts réellement réalisés pour la distribution BinomialUNOS : 82.82844088176348
Pourcentage de transferts réellement réalisés pour la distribution BinomialUNOS : 52.65964912280701
----------------------
Nombre moyen de transferts réellement réalisés pour la distribution BinomialAPD : 145.896
Variance des transferts réellement réalisés pour la distribution BinomialAPD : 155.35589579158318
Pourcentage de transferts réellement réalisés pour la distribution BinomialAPD : 42.6596491228

## Question 3

In [4]:
include("dynamic_process.jl")
G, edge_weight, gsm, gsd, pra = read_dat_file("data_KEP/KEP_191.dat");
total_achieved, total_choosen = stochastic_process(G,pra; distribution = "Binomial", verb=1)

println("** Simulation over 1 year with update each month **")
println("\t-> Total transfers achieved : $total_achieved out of $total_choosen cycles initialy choosen" )

Initial waiting list : [2, 3, 4, 6, 11, 16, 18, 19, 20, 23, 27, 28, 32, 33, 37, 38, 40, 43, 44, 46, 51, 52, 53, 55, 56, 59, 61, 62, 66, 67, 69, 72, 74, 75, 77, 78, 80, 81, 83, 84, 85, 90, 92, 95, 97, 99, 102, 103, 104, 107, 108, 111, 112, 114, 115, 116, 118, 120, 122, 123, 124, 126, 129, 131, 133, 134, 136, 139, 140, 141, 142, 143, 147, 148, 150, 151, 155, 156, 159, 160, 162, 164, 166, 170, 171, 173, 175, 178, 182, 185, 189, 190, 191, 192, 193, 194, 195, 197, 198, 201, 203, 204, 205, 207, 208, 209, 211, 216, 221, 222, 225, 232, 236, 238, 241, 242, 243, 248, 250, 253, 255, 256, 260, 263, 266, 267, 269, 273, 276, 278, 279, 280, 281, 283, 284, 285, 286, 287, 289, 291, 292, 298, 300, 303, 304, 305, 306, 309, 310, 311, 312, 317, 322, 329, 331, 332, 334, 336, 337, 338, 339, 340, 342, 343, 346, 347, 348, 351, 352, 354, 356, 359, 361, 363, 364, 367, 368, 370, 372, 373, 374, 375, 377, 378, 380, 386, 387, 389, 390, 392, 394, 395, 396, 398, 399, 402, 406, 408, 410, 413, 414, 416, 418, 420, 423, 4

On remarque que la taille maximale des cycles influe peu sur le nombre de transferts effectivement réalisés. On peut en effet se dire qu'avec un cycle plus grand, on augmente la probabilité d'échec d'un transfert et donc l'échec du cycle. D'où le fait qu'avec K=3 ou K=4, les pourcentages de réalisation sont légèrement plus faible qu'avec K=2 car il y a plus de cycles possibles mais on a quasiment toujours le même nombre de transferts effectivement réalisés.


## Question 4

Au vue de la question précédente, nous choisissons d'effectuer les tests suivants avec seulement $K=2$.

In [11]:
G, edge_weight, gsm, gsd, pra = read_dat_file("data_KEP/KEP_191.dat");
nb_simul = 10

println("** $nb_simul simulations of determinist and maximum 
            expectation approach on a 1 year (12 steps) dynamic
            process for various compatibility distribution      **")

failure_laws = ["Constant", "Binomial", "BinomialUNOS"]

for distribution in failure_laws

    success_base = zeros(Int64, nb_simul); choosen_base = zeros(Int64, nb_simul)
    success_maxE = zeros(Int64, nb_simul); choosen_maxE = zeros(Int64, nb_simul)
    for i in 1:nb_simul
        success_base[i], choosen_base[i] = stochastic_process(G,pra; K=2, sto_method="Determinist", distribution = distribution)
        success_maxE[i], choosen_maxE[i] = stochastic_process(G,pra; K=2, sto_method="Max-Expectation", distribution = distribution)
    end

    println("-----------------------------------")
    println("* $distribution *")
    println("-----------------------------------")
    println("• Determinist approach : ")
    println("\t - average transfers achieved     : $(mean(success_base))")
    println("\t - standard deviation             : $(var(success_base))")
    println("\t - average pourcentage of achieved: $(100*mean(success_base)/mean(choosen_base))")
    println("• Maximum Expectation approach : ")
    println("\t - average transfers achieved     : $(mean(success_maxE))")
    println("\t - standard deviation             : $(var(success_maxE))")
    println("\t - average pourcentage of achievid: $(100*mean(success_maxE)/mean(choosen_maxE))")

end

** 10 simulations of determinist and maximum 
            expectation approach on a 1 year (12 steps) dynamic
            process for various compatibility distribution      **
-----------------------------------
* Constant *
-----------------------------------
• Determinist approach : 
	 - average transfers achieved     : 121.2
	 - standard deviation             : 73.06666666666666
	 - average pourcentage of achieved: 8.945969884853854
• Maximum Expectation approach : 
	 - average transfers achieved     : 118.6
	 - standard deviation             : 83.60000000000001
	 - average pourcentage of achievid: 9.054817529393802
-----------------------------------
* Binomial *
-----------------------------------
• Determinist approach : 
	 - average transfers achieved     : 118.4
	 - standard deviation             : 196.26666666666668
	 - average pourcentage of achieved: 8.690546095126248
• Maximum Expectation approach : 
	 - average transfers achieved     : 183.4
	 - standard deviation        

On remarque tout d'abord qu'il y a environ les mêmes résultats pour les deux méthodes ("déterministe" et Maximum d'Espérance) pour la loi d'incompatibilité *constante*. C'est logique car étant donné que toutes probabilités d'incompatibilité sont égales à $0.7$, le problème résolu par le modèle de Maximum d'Espérence est identique à celui déterminite. Il y a juste un facteur $0.7^K = 0.49$ sur l'objectif du modèle. Ce qui ne change pas du tout les solutions produites. 

Pour la loi d'imcopatibilité *binomiale* basique, on observe en moyenne $\simeq 50$ % plus de transferts réalisés par la maximisation d'espérance par le modèle basique. De plus, le modèle de maximisation d'espérence est beaucoup plus efficace car environ la moitié des transferts qu'il sélectionne sont effectivement réalisables. Ce qui est beaucoup une fois comparé à la méthode "déterministe" dont seulement $\simeq 8$ % (soit 6 à 7 fois moins) des transferts sélectionnés sont effectivement réalisables.  

Pour la *Biomiale UNOS* qui est très particulière et s'intérèsse à la sensibilité "UNOS" des patiens, on observe que les résultats sont que très légèrement meilleurs pour le modèle de maximisation d'espérance. C'est peut-être du au fait qu'il y ait seulement deux valeurs possibles de probabilité, donc pour $K=2$ que trois valeurs possible d'espérance à associer aux cycles. Ce qui fait approcher ce problème du problème "constant". 



## Question 5 


Une première observation très naïve que l'on pourrait faire est que l'ajout des incertitude, bien que nécessaire pour approcher un problème concret, diminue fortement le nombre de transfert réalisés. Ainsi on obsrve une prologation des temps d'attente pour les paires patient/donneur en liste d'attente par rapport au cas déterministe (mais irréaliste).

L'utilisation du modèle de maximisation de l'espérance permet potentiellement un gain de faisabilité des cycles choisis. Nous avons vu que ce gains dépendait fortement de la modélisation de la loi d'incompatiblité. La modélisation *binomiale* semble très intéressante car c'est la plus efficace pour augmenter le nombre de transfert réalisables. D'un point de vue moral, ce choix a l'avantage de mettre tous les patients sur un même pied d'égalité. Mais cette "égalité" est en réalité une "égalité" par rapport au hasard. 

Il y a un piège dans cette modélisation. On sait que lorsque l'on connait un approximation des "probabilités" d'incompatibilité des patients, nous allons pouvoir utiliser ces informations pour choisir des cycles qui ont plus de chance d'être réalisé. Et donc au final, réaliser plus de transfert sur une période donnée. Cependant, cette modélisation invente simplement une probabilité.  

En fait cette modélisation *binomiale* revient simplement à accorder une priorité à certains patients, plutôt qu'à d'autres, de manière aléatoire. Et à la fin les vrais tests d'incompatibilités seront totalement en désaccord avec les probilités données, donc la solution n'aura pas plus, voir parfois moins d'intérêt que celle obtenue par le modèle déterministe.  

Les deux dernière modélisation : *binomiale UNOS* et *binomiale APD* se basent sur une vraie information médicale et donc peuvent (en théorie) augmenter le nombre de transfers réalisés à l'année. Il faudrait questionner cependant le choix des seuils utilisés ($0.8$ ou $0.75$) sur ces données de sensibilités. En effet, un patient avec une sensibilité APD de $0.79$ recoit une probabilité d'incompatilibité de $0.1$. Mais un patient avec une sensibilité APD légèrement plus élevée ($0.81$) va se voir affecter une probabilité d'échec de $0.9$. 

Avec ce choix, la vie de patient peut reposer sur une incertitude de mesure de ses données médicales. Le danger est très visible ici mais c'est en fait un problème qui peut être généralisé à n'importe quel modélisation de probabilité qui reposerait sur les données de santé des patients. 

## Question 6 


La méthode de recours décrite permet d'inclure, dans le calcul d'espérance des cycles, l'espérance des sous-cycles. De cette manière, les cycles choisis au premiers niveaux ont plus de chance de contenir des sous-cycles toujours réalisables après les résultats des tests croisés. 

D'un point de vue sociétal, cela va favoriser les patients qui des plusieurs compabilités (élevées) avec les autres donneurs du programme. Si tenté que la modélisation de compatibilité repose sur des données médicales. 

Nous pouvons attendre de cette méthode qu'elle re-donne un avantage à l'exploration de cycles de taille supérieure à $2$. En effet, plus un cycle est grand, plus il peut contenir des sous-cycles et donc renforcer son espérance. Cela dépend de la structures (de compatibilité primaires) et des valeurs de probabilité de chaque cycle. 